# New high-altitude dataset: independent review

Development split only. Review saved results and raw drone images; no fitting, threshold search or locked-test evaluation. Existing result configurations remain separate. Notebook outputs are the analysis record.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
root = Path.cwd()
while not (root / "single_photo_pipeline.py").exists():
    if root == root.parent:
        raise RuntimeError("Repository not found")
    root = root.parent
folder = root / "tests/Review_Highalt_20261118_codex"
output = folder / "results"
sys.path.insert(0, str(folder))
from review_blocks import *
source = root / "tests/Test_Iteration_12/results"
images = root / "dataset/Dataset_Official_3/d1/images"
development, metadata, split = load_development(images / "metadata.csv", source / "highalt_split.csv")
print("All metadata:", len(metadata), "Development:", len(development), "Locked images excluded:", int((split.split != "dev").sum()))
print("Height range:", metadata.alt.min(), metadata.alt.max(), "median", metadata.alt.median())
display(development.groupby("band", observed=True).alt.agg(["size", "min", "max"]))

In [ ]:
stage_path = next((source / "stageeval/tables_highalt_dev332").glob("per_photo*.csv"))
stages = pd.read_csv(stage_path).sort_values("photo").reset_index(drop=True)
assert np.allclose(stages.height, development.alt, atol=1e-5)
assert len(stages) == len(development)
summary = summarise_stages(stages)
summary.to_csv(output / "stage_by_height_codex.csv", index=False)
display(summary.round(3))
print("Single-query K100/M25 score", 6 * stages.points.mean(), "wrong", int((stages.points < 0).sum()))
headings = pd.read_csv(next((source / "stageeval/tables_highalt_dev332").glob("headings_per_photo*.csv")))
comparison = []
for variant, group in headings.groupby("variant", sort=False):
    comparison.append(dict(variant=variant, photos=len(group), score60=6*group.points.mean(), wrong=int((group.points < 0).sum()), answered=int(group.answered.sum())))
display(pd.DataFrame(comparison))
pd.DataFrame(comparison).to_csv(output / "query_variants_codex.csv", index=False)

In [ ]:
samples = select_visual_samples(development)
samples.to_csv(output / "viewed_development_images_codex.csv", index=False)
figure, axes = plt.subplots(7, 2, figsize=(10, 20))
for index, row in samples.iterrows():
    raw, view = read_view(images / row.filename)
    axis = axes.flat[index]
    axis.imshow(view, cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"{row.alt:.0f} m | {row.filename[:8]} | {row.band}")
    axis.axis("off")
figure.suptitle("Development drone images — independent per-image 1–99% stretch", fontsize=14)
figure.tight_layout(rect=(0, 0, 1, .98))
figure.savefig(output / "drone_images_by_height_codex.png", dpi=140)
plt.show()

In [ ]:
focal = float(development.cam_fx.median())
physics = pd.DataFrame([sample_physics(h, focal) for h in [200, 500, 800, 1000, 1400, 1600, 2000, 2500, 3000]])
physics.to_csv(output / "resolution_by_height_codex.csv", index=False)
display(physics.round(2))
print("Nominal level11 cap starts limiting footprint selection above", 224*.3*2**(11/2)*focal/320, "metres; ignores tilt/yaw/height-scale corrections")
figure, axes = plt.subplots(1, 2, figsize=(12,4))
summary.plot(x="band", y=["retrieval_share", "kept_share"], kind="bar", ax=axes[0], ylim=(0,1), title="Single-query retrieval vs retained candidates")
summary.assign(answered_share=summary.answered/summary.photos).plot(x="band", y="answered_share", kind="bar", ax=axes[1], ylim=(0,1), title="Accepted share (single query)")
figure.tight_layout()
figure.savefig(output / "failure_stages_by_height_codex.png", dpi=150)
plt.show()

In [ ]:
refinements = []
for name in ["highalt_dev332", "official500", "nudt300"]:
    table = pd.read_csv(source / f"coarse_refine/refine_{name}.csv")
    best = table.loc[table.groupby("photo").z1.idxmax()]
    groups = [("all", best)]
    if name == "highalt_dev332":
        groups.append(("height>=800", best[best.height>=800]))
    for group_name, group in groups:
        for variant in ["c", "p", "f2p", "f4p"]:
            if "err_"+variant in group:
                refinements.append(dict(data=name, group=group_name, variant=variant, **summarise_coarse(group, "err_"+variant)))
refinement_summary = pd.DataFrame(refinements)
display(refinement_summary.round(3))
refinement_summary.to_csv(output / "coarse_refinement_rescore_codex.csv", index=False)
confidence = []
for name in ["highalt_dev332", "official500", "nudt300"]:
    table = pd.read_csv(source / f"coarse_confidence/photos_{name}.csv")
    wrong = table[table.err>8]
    confidence.append(dict(data=name, photos=len(table), wrong=len(wrong), maximum_z_among_wrong=float(wrong.z.max())))
display(pd.DataFrame(confidence))
pd.DataFrame(confidence).to_csv(output / "coarse_confidence_transfer_codex.csv", index=False)
truth = pd.read_csv(source / "highalt_truth_diag/var_base.csv")
assert set(truth.filename).issubset(set(development.filename))
truth_bands = []
for band, group in add_height_band(truth, "alt").groupby("height_band", observed=True):
    truth_bands.append(dict(band=str(band), photos=len(group), truth_has_zero_inliers=int((group.inl_truth==0).sum()), truth_in_M=float(group.truth_in_M.mean()), median_truth_inliers=float(group.inl_truth.median())))
display(pd.DataFrame(truth_bands))
pd.DataFrame(truth_bands).to_csv(output / "truth_known_matcher_by_height_codex.csv", index=False)

In [ ]:
paths = [images / "metadata.csv", source / "highalt_split.csv", stage_path]
paths += list((source / "coarse_refine").glob("*.csv"))
paths += list((source / "coarse_confidence").glob("photos_*.csv"))
paths += [images / name for name in samples.filename]
manifest = []
for path in paths:
    manifest.append(dict(path=str(path.relative_to(root)), sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
(output / "inputs_codex.json").write_text(json.dumps(manifest, indent=2))
print("Saved review tables, figures and input hashes. No weights fitted or thresholds selected.")